# Tutorial: The `DataSet` module

This tutorial covers `el_paso`'s `DataSet` module: loading data that was already
processed and saved (see `3_saving_strategies_and_datasets.ipynb`), the difference
between `DataSet` and its typed subclasses (`GFZDataSet`, `PRBEMDataSet`), a few
built-in analysis utilities (`interp_flux`, `interp_psd`, `identify_orbits`), how to
inspect variable metadata, and how to build a `DataSet` "by hand" from data that
never went through `el_paso`'s own saving pipeline.

In [ ]:
from datetime import datetime, timedelta, timezone
from pprint import pprint

import numpy as np
from matplotlib import pyplot as plt

import el_paso as ep
from el_paso.dataset import DataSet, GFZDataSet, PRBEMDataSet
from el_paso.dataset.interp_functions import TargetType

ep.setup_logging()

start_time = datetime(2017, 7, 14, tzinfo=timezone.utc)
end_time = datetime(2017, 7, 14, 23, 59, 59, tzinfo=timezone.utc)


# 0. Preparing RBSP data

For this tutorial, we need fully processed data. For this, we can use EL-PASO's recipes.

In [ ]:
from el_paso.recipes.rbsp import process_rbsp_hope_electrons

process_rbsp_hope_electrons(
    start_time,
    end_time,
    sat_str="a",
    mag_field="T89",
    raw_data_path=".",
    processed_data_path=".",
    num_cores=12,
    save_strategy="netcdf",
    calculate_Lstar=False,
)

## 1. Loading processed data with `DataSet`

`el_paso.dataset.DataSet` reads data back from disk using the same `SavingStrategy`
object that was used to write it. To load the `MonthlyRBStrategy` + `GFZStandard`
data, we first re-create the exact same strategy (pointing at the
same `base_data_path`, `mission`, `satellite`, `instrument` and `mag_field`), and then
hand it to `DataSet`.


In [ ]:
saving_strategy = ep.saving_strategies.MonthlyRBStrategy(
    ".",
    mission="RBSP",
    satellite="rbspa",
    instrument="hope",
    mag_field="T89",
    data_standard=ep.data_standards.GFZStandard(),
)

data_set = DataSet(saving_strategy, start_time, end_time)
data_set


Creating the `DataSet` does **not** touch disk at all. Data is loaded lazily, one variable at a time, the first time you access it.


In [ ]:
# First access triggers the actual file read
data_set.xGEO.shape


In [ ]:
# When the first variable is loaded, the time variable is loaded as well. For convinience, it is automatically converted to datetime objects as well and ready to access
pprint(data_set.time[:3])
pprint(data_set.datetime[:3])


`possible_variables` lists everything that *could* be loaded for this data standard
(plus a couple of computed properties, see below), regardless of whether it was
actually saved:


In [ ]:
data_set.possible_variables


Typos are also handled explicitly. `DataSet` keeps `__getattr__`/`__setattr__` restricted to `possible_variables`, and suggests the closest valid name when you get it wrong:


In [ ]:
try:
    data_set.xGRO  # typo
except AttributeError as e:
    print(e)


`DataSet` also exposes a couple of **computed** properties that are derived from
other variables rather than read from disk:

- `P`: azimuth in radians, computed from `MLT`.
- `InvV`: third adiabatic invariant, computed from `InvMu` and `InvK`.

Both are computed (and cached) automatically the first time they're accessed, as
long as the variables they depend on are available.


The code we have used so far is specific for the `GFZDataStandard`. The attribute access to variable `xGEO` might fail for other data sets, which are storred using a different data standard, as the `xGEO` variable might not exist. In order to write data-standard-agnostic code, the function `get_by_internal_name` must be used, which takes as input not the variable name as defined by the data standard, but the EL-PASO internal name instead.

In [ ]:
data_set.get_by_internal_name("Position").shape # same as gfz_data.xGEO but works for any data standard

## 2. `GFZDataSet` / `PRBEMDataSet`: same runtime behaviour, better type hints

`GFZDataSet` and `PRBEMDataSet` are thin subclasses of `DataSet`. They exist for one
reason: their class body annotates every variable defined by `GFZStandard` /
`PRBEMStandard` (e.g. `xGEO: NDArray[np.float64]`), which is what gives you IDE
autocompletion and static type checking on `ds.xGEO`, `ds.PSD`, etc. `DataSet` itself
has none of these annotations, so autocompletion won't suggest variable names for it.


In [ ]:
gfz_data_set = GFZDataSet(saving_strategy, start_time, end_time)
gfz_data_set.xGEO.shape  # now autocompletes as `xGEO` in an editor/IDE

## 3. Utility functions: `interp_flux`, `interp_psd`, `identify_orbits`

`DataSet` bundles a few analysis helpers as methods.

### `interp_flux`

Interpolates `FEDU` in energy (at the two pitch angles from `Alpha_Eq` that bracket
each target pitch angle), then linearly across pitch angle. `target_type` controls
how `target_en`/`target_al` are combined: `TargetType.TargetPairs` interpolates each
`(energy, pitch angle)` pair (equal-length vectors), `TargetType.TargetMeshGrid`
interpolates every combination.

In [ ]:
target_en = [10e-3, 30e-3] # in MeV
target_al = np.deg2rad([45.0, 70.0])

flux_pairs = data_set.interp_flux(target_en, target_al, target_type=TargetType.TargetPairs, n_threads=2)
flux_pairs.shape  # (n_time, len(target_en))


In [ ]:
flux_grid = data_set.interp_flux(target_en, target_al, target_type=TargetType.TargetMeshGrid, n_threads=2)
flux_grid.shape  # (n_time, len(target_en), len(target_al))


### `interp_psd`

Same idea, but for phase space density: interpolates `PSD` in `InvMu` at the two `K`
values (from `InvK`) bracketing each target `K`, then linearly across `K`.


In [ ]:
target_mu = [0.1, 5.0]
target_K = [0.1, 0.3]

psd_pairs = data_set.interp_psd(target_mu, target_K, target_type=TargetType.TargetPairs, n_threads=2)
psd_pairs.shape  # (n_time, len(target_mu))


### `identify_orbits`

Splits a time series into inbound/outbound trajectories based on the local minima and
maxima of a radial distance (`R_Eq` for `orbit_type="R"`, or the last column of
`L_star`/`Lstar` for `orbit_type="L*"`). Each `Trajectory` is a `(start, end,
direction)` tuple of indices into the time array.

We use a second, separate synthetic dataset here: an oscillating `R_Eq` that mimics a
satellite going through a few perigee/apogee passes.


In [ ]:
import matplotlib.dates as mdates
from matplotlib.lines import Line2D

trajectories = data_set.identify_orbits(orbit_type="R", minimal_distance=10)
pprint(trajectories[:3])

for traj in trajectories:
    idx = range(traj.start, traj.end+1)
    color = "r" if traj.direction == "inbound" else "g"
    plt.plot(np.asarray(data_set.datetime)[idx], data_set.R0[idx], color)

inbound_handle = Line2D([], [], color="r", label="Inbound")
outbound_handle = Line2D([], [], color="g", label="Outbound")
plt.legend(handles=[inbound_handle, outbound_handle])

ax = plt.gca()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())

plt.tight_layout()


## 4. Reading metadata

Every `DataSet` carries a `.metadata` attribute that mirrors the dataset's
variables. Accessing `ds.metadata.<name>` returns a `VariableMetadata` describing
`unit`, `original_cadence_seconds`, `source_files`, `description`,
`processing_notes` and `standard_name`.


In [ ]:
data_set.metadata.Flux


`to_dict()` (an alias of `as_dict()`) returns metadata for every **currently loaded**
variable as a plain dict -- variables that haven't been accessed yet simply won't
appear:


In [ ]:
data_set.metadata.to_dict()


## 5. Building a "mock" `DataSet` for external code

Sometimes another codebase (a model, an assimilation tool, a plotting library you
didn't write) expects an object with the same interface as `ep.DataSet` (attributes
like `.PSD`, `.Lstar`, `.datetime` following a particular data standard) as input.
But your actual data might come from somewhere that never went through `ep.save`: a
one-off analysis, a different mission's custom processing pipeline, a live
simulation, etc.

Rather than reimplementing that interface yourself, you can construct a `DataSet`
(or a typed `GFZDataSet`/`PRBEMDataSet`, for autocompletion) bound to a saving
strategy that is **never actually used for I/O**, and set your own arrays on it
directly. `DataSet.__setattr__` still validates every name against
`possible_variables` for the chosen data standard, so you get the same typo
protection as when loading real data.


In [ ]:
mock_strategy = ep.saving_strategies.MonthlyRBStrategy(
    base_data_path=".",  # never actually read from or written to below
    mission="mock",
    satellite="mock",
    instrument="mock",
    mag_field="T89",
    data_standard=ep.data_standards.GFZStandard(),
)

mock_ds = GFZDataSet(mock_strategy, start_time, end_time)

n = 4
mock_ds.datetime = [start_time + timedelta(minutes=10 * i) for i in range(n)]
mock_ds.Lstar = np.array([[4.1], [4.3], [4.5], [4.2]])
mock_ds.PSD = np.array([1.2e-6, 1.5e-6, 1.1e-6, 0.9e-6])
mock_ds.MLT = np.array([1.0, 3.0, 6.0, 9.0])


This can now be passed to any function written against the `DataSet` interface:


In [ ]:
def summarize(ds: DataSet) -> None:
    print(f"{len(ds.datetime)} time steps, PSD range: {ds.PSD.min():.2e} - {ds.PSD.max():.2e}")


summarize(mock_ds)


Computed properties still work exactly as before: `P` is derived from `MLT` the
first time it's accessed, even though `mock_ds` was never loaded from a file:


In [ ]:
mock_ds.P


## Further reading

Two more advanced `DataSet` methods weren't covered here:

- `linearize_trajectories`: maps a time series onto a linearized radial-distance
  axis using the trajectories from `identify_orbits` (as in Haas et al. papers).
- `bin_and_interpolate_to_model_grid`: bins and interpolates a variable (e.g. `PSD`)
  onto a radiation-belt model's V-K, spatial and time grids, for use as assimilation
  input.
